In [15]:
from typing import Any
from langchain.agents.middleware import (AgentMiddleware, AgentState, hook_config)
from langchain.agents.middleware import (PIIMiddleware, HumanInTheLoopMiddleware)
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool 
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import AIMessage

class HealthcareSafetyFilter(AgentMiddleware):
    """Block non medical or harmfull request"""
    Blocked_topic=[
        "Drug_synthesis", "self-harm", "suicide method", "weapon", "hack"
    ]
    @hook_config(can_jump_to=["end"])
    def before_agent_call(self, state: AgentState, runtime: Runtime)-> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_msg=state["messages"][0]
        if first_msg.type != "human":
            return None
        
        content= first_msg.content.lower()
        for topic in self.Blocked_topic:
            if topic.lower in content:
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I am a healthcare assistant and can only help"
                            "with medical question, appointments and health information"
                            "if you are in crisis, please call 1122 or your local emergency number"
                        )
                    }],
                    "jump_to": "end"
                }
        return None

# Medical Output Validator

In [16]:
class MedicalOutputValidator(AgentMiddleware):
    "Ensure all responses include appropriate medical disclaimers"
    DISCLAIMER= (
        "\n\n This is general health information, not medica advice."
        "Please consult a qualified healthcare professional"
    )

    @hook_config(can_jump_to=["end"])

    def after_agent(self, state: AgentState, runtime: Runtime)-> dict[str, Any] | None:
        if not state["messages"]:
            return None
        
        last_msg=state["messages"][-1]
        if not isinstance(last_msg, AIMessage):
            return None
        
        # Add disclaimer if already not present 
        if "medical advice" not in last_msg.content.lower():
            last_msg.content+= self.DISCLAIMER
        
        return None


# HealthCare Tool 

In [17]:
@tool
def search_symptoms(symptoms: str)-> str:
    """Search for information about medical symptoms"""
    return (
        f"Symptoms information for : {symptoms}"
        "Please Consult a doctor for diagnosis")
    
@tool
def book_appointment(patient_name: str, date: str, doctor: str)-> str:
    """Book the medical appointment"""
    return (
        f"Appointment booked for {patient_name}"
        f"With Dr. {doctor} on {date}"
    )
@tool 
def get_medication_info(medication: str)-> str:
    """Get information about a medication."""
    return (
        f"General info about {medication}."
        "Always follow your doctor prescription"
    )

# Assembling HealthCare Chatbot

In [18]:

healthcare_bot=create_agent(
    model="gpt-4o-mini",
    tools=[search_symptoms, book_appointment, get_medication_info],
    middleware=[
        # Guardrail 1: Block Harmfull/off-topic request
        HealthcareSafetyFilter(),

        # Guardrail 2: Redact patient PII from inputs
        PIIMiddleware(
            "email", strategy="redact", apply_to_input=True
        ),
        PIIMiddleware(
            "credit_card", strategy="redact", apply_to_input=True
        ),

        # guardrail 3: Require approval before booking appointment
        HumanInTheLoopMiddleware(
            interrupt_on={
                "book_appointment": True,
                "search_symptoms" : False,
                "get_medication_info": False,
            }
        ),

        # Guardrail 4: Add medical disclaimer to all outputs
        MedicalOutputValidator(),
    ],
    checkpointer= InMemorySaver(),
    system_prompt=(
        "you are a helpfull healthcare assistant"
        "you can search for symptoms, medication information,"
        "and help to book appointment. Always be empathetic and"
        "remind user to consult a doctor for diagonsis"
    ),
)
print("Healthcare chatbot with full guardrail stack created!")

Healthcare chatbot with full guardrail stack created!


# Testing the healthcare chatbot 

In [21]:
config_t1={"configurable":{"thread_id": "healthcare_session_t1"}}

result= healthcare_bot.invoke(
    {"messages":[{"role": "user", "content": "what are the sympthoms of type 2 Diabetes?"}]},
    config=config_t1
)
print(result["messages"][-1].content)

Symptoms of type 2 diabetes can vary but may include:

- Increased thirst
- Frequent urination
- Extreme fatigue
- Blurred vision
- Slow-healing wounds or infections
- Areas of darkened skin, often in the armpits and neck

It's really important to consult a doctor for an accurate diagnosis and personalized advice.

 This is general health information, not medica advice.Please consult a qualified healthcare professional


# Query with PII

In [22]:
result=healthcare_bot.invoke(
    {"messages":[{"role":"user", "content":(
        "My email is patient123@gmail.com"
        "what can i take for headache"
    )}]},
    config=config_t1
)
print("==PII Redaction Test==")
print(result["messages"][-1].content)

==PII Redaction Test==
For headaches, common over-the-counter medications include:

- Acetaminophen (Tylenol)
- Ibuprofen (Advil, Motrin)
- Aspirin

However, it's essential to follow your doctor's prescription or recommendations when taking any medication. If your headaches persist or worsen, please consult a doctor for further advice.

 This is general health information, not medica advice.Please consult a qualified healthcare professional


In [23]:
result=healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "how to i synthesize drugs at home?"}]},
    config=config_t1
)
print("==Blocked request==")
print(result["messages"][-1].content)

==Blocked request==
I'm sorry, but I cannot assist with that. Synthesizing drugs at home can be extremely dangerous and is illegal in many places. It's important to only use medications that are prescribed by a qualified healthcare professional. If you have questions or concerns about medications or their uses, please consult a doctor or pharmacist for safe and reliable guidance.

 This is general health information, not medica advice.Please consult a qualified healthcare professional


# Appointment booking (Human approvel)

In [27]:
config={"configurable":{"thread_id":"healthcare_session_001"}}
result=healthcare_bot.invoke(
    {"messages":[{"role":"user", "content":"am shayan Book the appointment for me with dr abdullah on 12 march"}]},
    config=config
)
print("==Appointment Booking --Awaiting approval==")

#Approve
from langgraph.types import Command
approved=healthcare_bot.invoke(
    Command(resume={"decisions":[{"type":"approve"}]}),
    config=config
)
print("==After Approval==")
print(approved["messages"][-1].content)

==Appointment Booking --Awaiting approval==
==After Approval==
Your appointment with Dr. Abdullah has been successfully booked for March 12th. If you have any further questions or need assistance, feel free to ask! And remember, it's always a good idea to consult with a doctor for any medical concerns.

 This is general health information, not medica advice.Please consult a qualified healthcare professional


In [25]:
#Approve
from langgraph.types import Command
approved=healthcare_bot.invoke(
    Command(resume={"decisions":[{"type":"approve"}]}),
    config=config
)
print("==After Approval==")
print(approved["messages"][-1].content)

==After Approval==
Could you please provide me with your name and the date you would like to book the appointment with Dr. Abdullah?
